In [1]:
import os
import cv2
import zipfile
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization,
    Input, GlobalAveragePooling2D, Rescaling, Lambda
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## 1. CONFIGURATION
## ==========================================

In [3]:
# Dataset location. Override at runtime without editing the notebook, e.g.:
#   set DAISEE_ZIP_PATH=/path/to/Images.zip   (Windows)
#   export DAISEE_ZIP_PATH=/path/to/Images.zip  (Linux/macOS/WSL)
ZIP_PATH = os.environ.get(
    'DAISEE_ZIP_PATH',
    r'C:\Users\bhara\OneDrive\Desktop\final_year_project\dataset\Images.zip'
)
IMG_SIZE = (96, 96)  # Standardized size for network input
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42
OUTPUT_DIR = 'outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ensure reproducibility
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Fail fast with a clear message instead of a confusing traceback later on
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Dataset zip file not found at: {ZIP_PATH}\n"
        f"Set the DAISEE_ZIP_PATH environment variable to the correct path, "
        f"or edit ZIP_PATH above."
    )

# Determine device (GPU/CPU)
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPU is available! Using: {physical_devices}")
    for device in physical_devices:
        try:
            tf.config.experimental.set_memory_growth(device, True)
        except Exception as e:
            print(f"Could not set memory growth for {device}: {e}")
else:
    print("No GPU found. Falling back to CPU gracefully.")

No GPU found. Falling back to CPU gracefully.


## 2. DATASET INSPECTION & PREPROCESSING
## ==========================================

In [4]:
def extract_and_preprocess_from_zip(zip_path, img_size):
    print("\n--- Inspecting and Preprocessing Dataset ---")
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Dataset zip file not found at {zip_path}")

    images = []
    labels = []
    class_names = []
    original_shapes = []

    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    with zipfile.ZipFile(zip_path, 'r') as z:
        file_list = [f for f in z.namelist() if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        print(f"Found {len(file_list)} image files in zip.")

        for f in file_list:
            class_name = os.path.basename(os.path.dirname(f))
            if not class_name:
                # Images sitting at the zip root with no class folder — skip, they
                # can't be labeled.
                continue
            if class_name not in class_names:
                class_names.append(class_name)

            # Read image from zip into memory
            img_data = z.read(f)
            img_array = np.frombuffer(img_data, np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

            if img is None:
                continue

            original_shapes.append(img.shape)

            # Face Detection / Cropping
            # If the image is large, we attempt face detection.
            # If it's already small (e.g., < 150x150), assume it's already cropped faces.
            if max(img.shape[0], img.shape[1]) > 150:
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(30, 30))

                if len(faces) > 0:
                    # Take the largest detected face, not just the first one returned
                    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
                    face_img = img[y:y + h, x:x + w] if w > 0 and h > 0 else img
                else:
                    face_img = img
            else:
                face_img = img

            # Resize (normalization is left to the model itself, see Section 4 —
            # this keeps a single preprocessing path that works for both the
            # custom CNN and the MobileNetV2 transfer model)
            face_img = cv2.resize(face_img, img_size)
            face_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)

            images.append(face_img)
            labels.append(class_names.index(class_name))

    if len(images) == 0:
        raise ValueError(
            "No images were loaded from the zip. Check that the zip contains "
            "images organized as <class_name>/<image>.jpg and that ZIP_PATH is correct."
        )

    print(f"\nSample of original image shapes: {list(set(original_shapes[:10]))}")

    # Check class distribution
    unique_labels, counts = np.unique(labels, return_counts=True)
    print("\nClass Distribution:")
    for lbl, count in zip(unique_labels, counts):
        print(f"  {class_names[lbl]}: {count}")

    # Keep images as float32 in [0, 255]. Rescaling to the range each model
    # expects happens inside the model itself (see Section 4).
    X = np.array(images, dtype='float32')
    y = np.array(labels)

    return X, y, class_names

X, y, class_names = extract_and_preprocess_from_zip(ZIP_PATH, IMG_SIZE)


--- Inspecting and Preprocessing Dataset ---
Found 2120 image files in zip.

Sample of original image shapes: [(720, 1280, 3)]

Class Distribution:
  Confused: 369
  Focused: 347
  Frustrated: 360
  Bored: 358
  Drowsy: 263
  Looking Away: 423


## 3. TRAIN/VAL/TEST SPLIT & DATA LEAKAGE CHECK
## ==========================================

In [5]:
print("\n--- Splitting Dataset ---")
# Because we read frames/images and they don't have distinct video identifiers
# (just 0020.jpg, 0021.jpg), we will use standard random split.
# ASSUMPTION: If there are distinct subjects/videos, random split might cause leakage.
# Given filenames like 0020.jpg, we assume they are independent images or we have no subject ID metadata.
# We will do a strict disjoint split of indices.

indices = np.arange(len(X))
# Train (70%), Val (15%), Test (15%)
X_temp, X_test, y_temp, y_test, idx_temp, idx_test = train_test_split(
    X, y, indices, test_size=0.15, stratify=y, random_state=SEED
)

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_temp, y_temp, idx_temp, test_size=0.1765, stratify=y_temp, random_state=SEED  # 0.1765 of 0.85 approx 0.15
)

# VERIFY NO DATA LEAKAGE between splits
train_set = set(idx_train)
val_set = set(idx_val)
test_set = set(idx_test)

if not train_set.isdisjoint(val_set) or not train_set.isdisjoint(test_set) or not val_set.isdisjoint(test_set):
    raise ValueError("DATA LEAKAGE DETECTED! Overlapping samples between splits.")
else:
    print("Data leakage check passed! Train, Val, and Test splits are strictly disjoint.")

print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}, Test samples: {len(X_test)}")

# Calculate Class Weights to handle imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# Data Augmentation (only applied during training, Keras skips it at inference automatically)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.1, seed=SEED),
    tf.keras.layers.RandomZoom(0.1, seed=SEED),
    tf.keras.layers.RandomBrightness(0.1, seed=SEED)
])


--- Splitting Dataset ---
Data leakage check passed! Train, Val, and Test splits are strictly disjoint.
Train samples: 1483, Validation samples: 319, Test samples: 318


## 4. MODEL BUILDING
## ==========================================

In [6]:
print("\n--- Building Models ---")

# Model A: Custom CNN
def build_custom_cnn(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = Rescaling(1. / 255)(x)  # inputs arrive as float32 in [0, 255]

    x = Conv2D(32, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(128, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name="Custom_CNN")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Model B: Transfer Learning (MobileNetV2)
def build_transfer_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs)
    # MobileNetV2 expects its own [-1, 1] preprocessing, applied to [0, 255] input.
    # Applying this instead of a plain /255 rescale is what the pretrained
    # ImageNet weights were trained on, and skipping it silently hurts accuracy.
    x = Lambda(mobilenet_preprocess)(x)

    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze base

    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name="MobileNetV2_Transfer")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

num_classes = len(class_names)
cnn_model = build_custom_cnn((IMG_SIZE[0], IMG_SIZE[1], 3), num_classes)
tl_model = build_transfer_model((IMG_SIZE[0], IMG_SIZE[1], 3), num_classes)

cnn_model.summary()
tl_model.summary()


--- Building Models ---

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 94, 94, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 45, 45, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 20, 20, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 20, 20, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,733,446 (6.61 MB)

 Trainable params: 1,732,998 (6.61 MB)

...

Model: "MobileNetV2_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 5. TRAINING
## ==========================================

In [7]:
print("\n--- Training Custom CNN ---")
callbacks_cnn = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=3),
    ModelCheckpoint(os.path.join(OUTPUT_DIR, 'cnn_best.keras'), save_best_only=True)
]
history_cnn = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks_cnn,
    verbose=1
)

print("\n--- Training Transfer Learning Model ---")
callbacks_tl = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=3),
    ModelCheckpoint(os.path.join(OUTPUT_DIR, 'tl_best.keras'), save_best_only=True)
]
history_tl = tl_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks_tl,
    verbose=1
)


--- Training Custom CNN ---
Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - accuracy: 0.6305 - loss: 1.7796 - val_accuracy: 0.2696 - val_loss: 5.6340 - learning_rate: 0.0010
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 90ms/step - accuracy: 0.7350 - loss: 0.6843 - val_accuracy: 0.2006 - val_loss: 22.1313 - learning_rate: 0.0010
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - accuracy: 0.8139 - loss: 0.4861 - val_accuracy: 0.2006 - val_loss: 32.4292 - learning_rate: 0.0010
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - accuracy: 0.8051 - loss: 0.4768 - val_accuracy: 0.2006 - val_loss: 42.8169 - learning_rate: 0.0010
Epoch 5/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/step - accuracy: 0.8449 - loss: 0.3895 - val_accuracy: 0.2006 - val_loss: 58.9300 - learning_rate: 5.0000e-04
Epoch 6/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.8651 - loss: 0.3115 - val_accuracy: 0.2006 - val_loss: 49.5701 - learning_rate: 5.0000e-04

--- Training Transfer Learning Model ---
Epoch 1/20

## 6. EVALUATION
## ==========================================

In [8]:
print("\n--- Evaluating Models on Test Set ---")
cnn_loss, cnn_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
tl_loss, tl_acc = tl_model.evaluate(X_test, y_test, verbose=0)

print(f"Custom CNN Test Accuracy: {cnn_acc:.4f}")
print(f"Transfer Learning Test Accuracy: {tl_acc:.4f}")

# Select the best model
if tl_acc > cnn_acc:
    best_model = tl_model
    best_history = history_tl
    model_name = "Transfer Learning (MobileNetV2)"
    print("\nTransfer learning performed better!")
else:
    best_model = cnn_model
    best_history = history_cnn
    model_name = "Custom CNN"
    print("\nCustom CNN performed better!")

# Predict and plot for the best model
y_pred_probs = best_model.predict(X_test, batch_size=BATCH_SIZE)
y_pred = np.argmax(y_pred_probs, axis=1)

print(f"\nClassification Report ({model_name}):")
print(classification_report(y_test, y_pred, target_names=class_names))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig_cm = plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - {model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, bbox_inches='tight')
plt.close(fig_cm)
print(f"Saved {cm_path}")

# Plot accuracy and loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(best_history.history['accuracy'], label='Train')
ax1.plot(best_history.history['val_accuracy'], label='Val')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(best_history.history['loss'], label='Train')
ax2.plot(best_history.history['val_loss'], label='Val')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
plt.savefig(curves_path, bbox_inches='tight')
plt.close(fig)
print(f"Saved {curves_path}")


--- Evaluating Models on Test Set ---
Custom CNN Test Accuracy: 0.2767
Transfer Learning Test Accuracy: 0.9434

Transfer learning performed better!
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step

Classification Report (Transfer Learning (MobileNetV2)):
              precision    recall  f1-score   support

    Confused       0.95      1.00      0.97        55
     Focused       0.89      0.98      0.94        52
  Frustrated       1.00      0.87      0.93        54
       Bored       0.96      0.89      0.92        54
      Drowsy       0.86      0.93      0.89        40
Looking Away       0.98      0.98      0.98        63

    accuracy                           0.94       318
   macro avg       0.94      0.94      0.94       318
weighted avg       0.95      0.94      0.94       318

Saved outputs\confusion_matrix.png
Saved outputs\training_curves.png


## 7. SAVE ARTIFACTS
## ==========================================

In [9]:
model_save_path = os.path.join(OUTPUT_DIR, 'best_engagement_model.keras')
best_model.save(model_save_path)
print(f"\nSaved best model to {model_save_path}")

mapping_path = os.path.join(OUTPUT_DIR, 'class_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump({str(i): name for i, name in enumerate(class_names)}, f)
print(f"Saved class mapping to {mapping_path}")
print("Pipeline complete!")


Saved best model to outputs\best_engagement_model.keras
Saved class mapping to outputs\class_mapping.json
Pipeline complete!
